# 02 — Silver Layer: Data Transformation & Cleansing

**Purpose:** Read from Bronze, apply cleaning/enrichment, write to Silver Delta Tables (registered in Lakehouse).  
**Output:** `silver_sales`, `silver_customers`, `silver_products` Delta tables.  
**Schedule:** After notebook 01 completes.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, upper, lower, to_date, when, isnan, isnull,
    regexp_replace, coalesce, lit, round as spark_round,
    year, month, dayofmonth, dayofweek, date_format,
    current_timestamp, monotonically_increasing_id
)
from pyspark.sql.types import DoubleType, IntegerType, StringType
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

LAKEHOUSE_NAME = "sales_lakehouse"
BASE = f"abfss://{LAKEHOUSE_NAME}@onelake.dfs.fabric.microsoft.com"
print("Spark session ready")

In [ ]:
# ─────────────────────────────────────────────
# 1. SALES — Read latest bronze partition
# ─────────────────────────────────────────────
df_bronze_sales = spark.read.format("delta").load(f"{BASE}/Files/bronze/sales")

# Get latest run_date only (incremental load)
latest_run = df_bronze_sales.selectExpr("max(_run_date) as max_date").first()["max_date"]
df_sales = df_bronze_sales.filter(col("_run_date") == latest_run)
print(f"Processing bronze/sales for run_date: {latest_run} | Rows: {df_sales.count():,}")

In [ ]:
# ─────────────────────────────────────────────
# 2. SALES — Cleanse & Transform
# ─────────────────────────────────────────────
df_silver_sales = (
    df_sales
    # Drop duplicates on natural key
    .dropDuplicates(["order_id", "order_line"])

    # Drop rows missing critical fields
    .dropna(subset=["order_id", "customer_id", "product_id", "order_date"])

    # Cast & clean columns
    .withColumn("order_date",   to_date(col("order_date"), "yyyy-MM-dd"))
    .withColumn("ship_date",    to_date(col("ship_date"),  "yyyy-MM-dd"))
    .withColumn("quantity",     col("quantity").cast(IntegerType()))
    .withColumn("unit_price",   col("unit_price").cast(DoubleType()))
    .withColumn("discount",     coalesce(col("discount").cast(DoubleType()), lit(0.0)))
    .withColumn("cogs",         col("cogs").cast(DoubleType()))
    .withColumn("region",       upper(trim(col("region"))))
    .withColumn("ship_mode",    trim(col("ship_mode")))
    .withColumn("order_id",     trim(col("order_id")))

    # Derived metrics
    .withColumn("gross_sales",  spark_round(col("quantity") * col("unit_price"), 2))
    .withColumn("discount_amt", spark_round(col("gross_sales") * col("discount"), 2))
    .withColumn("net_sales",    spark_round(col("gross_sales") - col("discount_amt"), 2))
    .withColumn("profit",       spark_round(col("net_sales") - col("cogs"), 2))

    # Filter out nonsensical rows
    .filter(col("quantity") > 0)
    .filter(col("unit_price") > 0)

    # Audit columns
    .withColumn("_silver_loaded_at", current_timestamp())

    # Select final columns
    .select(
        "order_id", "order_line", "order_date", "ship_date", "ship_mode",
        "customer_id", "product_id", "region",
        "quantity", "unit_price", "discount", "cogs",
        "gross_sales", "discount_amt", "net_sales", "profit",
        "_silver_loaded_at"
    )
)

print(f"Silver sales rows after cleaning: {df_silver_sales.count():,}")

In [ ]:
# ─────────────────────────────────────────────
# 3. CUSTOMERS — Cleanse & Transform
# ─────────────────────────────────────────────
df_bronze_customers = spark.read.format("delta").load(f"{BASE}/Files/bronze/customers")

df_silver_customers = (
    df_bronze_customers
    .dropDuplicates(["customer_id"])
    .dropna(subset=["customer_id", "customer_name"])
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("segment",       trim(upper(col("segment"))))
    .withColumn("country",       trim(col("country")))
    .withColumn("city",          trim(col("city")))
    .withColumn("state",         trim(col("state")))
    .withColumn("postal_code",   regexp_replace(col("postal_code"), "[^0-9]", ""))
    .withColumn("email",         lower(trim(col("email"))))
    .withColumn("_silver_loaded_at", current_timestamp())
    .select(
        "customer_id", "customer_name", "segment",
        "country", "state", "city", "postal_code", "email",
        "_silver_loaded_at"
    )
)

print(f"Silver customers: {df_silver_customers.count():,}")

In [ ]:
# ─────────────────────────────────────────────
# 4. PRODUCTS — Cleanse & Transform
# ─────────────────────────────────────────────
df_bronze_products = spark.read.format("delta").load(f"{BASE}/Files/bronze/products")

df_silver_products = (
    df_bronze_products
    .dropDuplicates(["product_id"])
    .dropna(subset=["product_id", "product_name"])
    .withColumn("product_name",  trim(col("product_name")))
    .withColumn("category",      trim(col("category")))
    .withColumn("sub_category",  trim(col("sub_category")))
    .withColumn("brand",         trim(col("brand")))
    .withColumn("standard_cost", col("standard_cost").cast(DoubleType()))
    .withColumn("list_price",    col("list_price").cast(DoubleType()))
    .withColumn("_silver_loaded_at", current_timestamp())
    .select(
        "product_id", "product_name", "category", "sub_category",
        "brand", "standard_cost", "list_price",
        "_silver_loaded_at"
    )
)

print(f"Silver products: {df_silver_products.count():,}")

In [ ]:
# ─────────────────────────────────────────────
# 5. Write Silver Tables (MERGE/UPSERT)
# ─────────────────────────────────────────────
def upsert_silver(df, table_name, merge_key):
    path = f"{BASE}/Tables/silver/{table_name}"
    try:
        dt = DeltaTable.forPath(spark, path)
        dt.alias("target").merge(
            df.alias("source"),
            f"target.{merge_key} = source.{merge_key}"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
        print(f"✅ MERGED into silver/{table_name}")
    except Exception:
        df.write.format("delta").mode("overwrite").save(path)
        print(f"✅ CREATED silver/{table_name}")

upsert_silver(df_silver_sales,     "sales",     "order_id")
upsert_silver(df_silver_customers, "customers", "customer_id")
upsert_silver(df_silver_products,  "products",  "product_id")

print("\n🎉 Silver transformation complete!")